# TubeWatch Transcript Storage Offline Tester

本 Notebook 只使用临时目录、正式 Python API 和可控 fake TubeScribe 结果，离线验证 schema、migration、transcript repository 与 processing 事务一致性。

In [1]:
from datetime import UTC, datetime, timedelta
from hashlib import sha256
from pathlib import Path
from tempfile import TemporaryDirectory
import sqlite3

from tubewatch import (
    AmbiguousTranscriptError, TubeScribeResult, VideoItem,
    cleanup_test_videos, get_transcript, initialize_state_database,
    list_transcripts_for_video, process_pending_videos,
    record_discovered_videos, save_transcript,
)
from tubewatch.exceptions import (
    StateStorageError, TubeScribeMembersOnlyError,
    TubeScribeNoSubtitlesError, TubeScribeProcessingError,
)

def video(video_id):
    return VideoItem(video_id, f'Title {video_id}', f'https://www.youtube.com/watch?v={video_id}', None, None, None)

def discover(db, source, *video_ids):
    return record_discovered_videos(
        db, source_type='channel', source_url=source,
        videos=[video(item) for item in video_ids], checked_at=datetime.now(UTC),
    )


In [2]:
# New schema, foreign keys, legacy upgrade, idempotence, and rollback.
legacy_schema = '''
CREATE TABLE sources (source_type TEXT NOT NULL, source_url TEXT NOT NULL, last_checked_at TEXT NOT NULL, PRIMARY KEY(source_type, source_url));
CREATE TABLE discovered_videos (source_type TEXT NOT NULL, source_url TEXT NOT NULL, video_id TEXT NOT NULL, title TEXT NOT NULL, url TEXT NOT NULL, published_at TEXT, channel_id TEXT, channel_name TEXT, first_seen_at TEXT NOT NULL, last_seen_at TEXT NOT NULL, PRIMARY KEY(source_type, source_url, video_id), FOREIGN KEY(source_type, source_url) REFERENCES sources(source_type, source_url) ON DELETE CASCADE);
CREATE TABLE processing_records (source_type TEXT NOT NULL, source_url TEXT NOT NULL, video_id TEXT NOT NULL, status TEXT NOT NULL CHECK(status IN ('pending','succeeded','failed','no_subtitles','members_only')), attempt_count INTEGER NOT NULL DEFAULT 0, last_attempt_at TEXT, raw_path TEXT, cleaned_path TEXT, language_code TEXT, is_automatic INTEGER, error_message TEXT, PRIMARY KEY(source_type, source_url, video_id), FOREIGN KEY(source_type, source_url,video_id) REFERENCES discovered_videos(source_type,source_url,video_id) ON DELETE CASCADE);
INSERT INTO sources VALUES ('channel','legacy','2025-01-01T00:00:00+00:00');
INSERT INTO discovered_videos VALUES ('channel','legacy','legacy-video','Legacy','https://example.invalid',NULL,NULL,NULL,'2025-01-01T00:00:00+00:00','2025-01-01T00:00:00+00:00');
INSERT INTO processing_records VALUES ('channel','legacy','legacy-video','succeeded',1,'2025-01-01T00:00:00+00:00','raw','cleaned','zh',0,NULL);
'''
with TemporaryDirectory() as temporary:
    root = Path(temporary)
    fresh = root / 'fresh.sqlite3'
    initialize_state_database(fresh)
    connection = sqlite3.connect(fresh)
    connection.execute('PRAGMA foreign_keys = ON')
    try:
        tables = {row[0] for row in connection.execute("SELECT name FROM sqlite_master WHERE type='table'")}
        assert {'videos', 'transcripts', 'schema_migrations'} <= tables
        assert connection.execute('PRAGMA foreign_keys').fetchone() == (1,)
        try:
            connection.execute("INSERT INTO transcripts (video_id,language_code,source_kind,cleaned_text,cleaned_content_hash,character_count,created_at,updated_at) VALUES ('missing','und','unknown','x','h',1,'t','t')")
        except sqlite3.IntegrityError:
            connection.rollback()
        else:
            raise AssertionError('transcript foreign key did not fire')
    finally:
        connection.close()

    legacy = root / 'legacy.sqlite3'
    connection = sqlite3.connect(legacy); connection.executescript(legacy_schema); connection.close()
    initialize_state_database(legacy); initialize_state_database(legacy)
    connection = sqlite3.connect(legacy)
    try:
        assert connection.execute('SELECT title FROM videos').fetchone() == ('Legacy',)
        assert connection.execute('SELECT status, transcript_id FROM processing_records').fetchone() == ('pending', None)
        assert connection.execute('SELECT COUNT(*) FROM schema_migrations').fetchone() == (2,)
    finally:
        connection.close()

    broken = root / 'broken.sqlite3'
    connection = sqlite3.connect(broken); connection.executescript(legacy_schema + 'CREATE TABLE videos (bad TEXT);'); connection.close()
    try:
        initialize_state_database(broken)
    except StateStorageError:
        pass
    else:
        raise AssertionError('broken migration unexpectedly succeeded')
    connection = sqlite3.connect(broken)
    try:
        assert connection.execute('SELECT status FROM processing_records').fetchone() == ('succeeded',)
        assert connection.execute("SELECT name FROM sqlite_master WHERE name='schema_migrations'").fetchone() is None
    finally:
        connection.close()
print('Schema and migration checks passed.')


Schema and migration checks passed.


In [3]:
# Transcript insert/read/upsert/hash/count/timestamps/multilingual/ambiguity/cascade.
with TemporaryDirectory() as temporary:
    db = Path(temporary) / 'repository.sqlite3'
    source = 'https://www.youtube.com/@repository/videos'
    discover(db, source, 'repo-video')
    first_time = datetime(2025, 1, 1, tzinfo=UTC)
    first = save_transcript(db, video_id='repo-video', language_code='zh', source_kind='manual', cleaned_text='第一版', raw_file_path='raw/repo.zh.vtt', saved_at=first_time)
    assert get_transcript(db, 'repo-video') == first
    assert first.cleaned_content_hash == sha256('第一版'.encode('utf-8')).hexdigest()
    assert first.character_count == len('第一版')
    unchanged = save_transcript(db, video_id='repo-video', language_code='zh', source_kind='manual', cleaned_text='第一版', raw_file_path='raw/repo.zh.vtt', saved_at=first_time + timedelta(hours=1))
    assert unchanged.id == first.id and unchanged.updated_at == first.updated_at
    updated = save_transcript(db, video_id='repo-video', language_code='zh', source_kind='manual', cleaned_text='第二版正文', raw_file_path='raw/repo.zh.vtt', saved_at=first_time + timedelta(hours=2))
    assert updated.id == first.id and updated.updated_at > first.updated_at
    save_transcript(db, video_id='repo-video', language_code='en', source_kind='auto_generated', cleaned_text='English')
    assert len(list_transcripts_for_video(db, 'repo-video')) == 2
    try:
        get_transcript(db, 'repo-video')
    except AmbiguousTranscriptError:
        pass
    else:
        raise AssertionError('ambiguous transcript selection was silent')
    assert get_transcript(db, 'repo-video', 'missing') is None
    connection = sqlite3.connect(db)
    connection.execute('PRAGMA foreign_keys = ON')
    try:
        connection.execute("DELETE FROM discovered_videos WHERE video_id='repo-video'")
        connection.execute("DELETE FROM videos WHERE video_id='repo-video'")
        connection.commit()
        assert connection.execute('SELECT COUNT(*) FROM videos').fetchone() == (0,)
        assert connection.execute('SELECT COUNT(*) FROM transcripts').fetchone() == (0,)
    finally:
        connection.close()
print('Transcript repository checks passed.')


Transcript repository checks passed.


In [4]:
# Offline fake TubeScribe success, idempotence, terminal outcomes, and DB rollback.
with TemporaryDirectory() as temporary:
    root = Path(temporary); db = root / 'processing.sqlite3'
    raw = root / 'output' / 'raw'; cleaned = root / 'output' / 'cleaned'
    raw.mkdir(parents=True); cleaned.mkdir(parents=True)
    source_one = 'https://www.youtube.com/@processing-one/videos'
    source_two = 'https://www.youtube.com/@processing-two/videos'
    discover(db, source_one, 'ok')

    def success_processor(url, *, raw_directory, cleaned_directory):
        video_id = url.split('=')[-1]
        raw_path = Path(raw_directory) / f'{video_id}.zh.vtt'
        cleaned_path = Path(cleaned_directory) / f'{video_id}.zh.txt'
        raw_path.write_text('WEBVTT\n', encoding='utf-8')
        cleaned_path.write_text('清理后的正文', encoding='utf-8')
        return TubeScribeResult(video_id, raw_path, cleaned_path, 'zh', False)

    first_batch = process_pending_videos(db, raw, cleaned, source_url=source_one, video_ids=['ok'], processor=success_processor)
    item = first_batch.results[0]
    assert item.status == 'succeeded' and item.transcript_saved and item.transcript_id
    assert item.raw_file_path == 'raw/ok.zh.vtt'
    transcript = get_transcript(db, 'ok')
    assert transcript and transcript.cleaned_text == '清理后的正文'
    assert transcript.character_count == len(transcript.cleaned_text)

    discover(db, source_two, 'ok')
    second_batch = process_pending_videos(db, raw, cleaned, source_url=source_two, video_ids=['ok'], processor=success_processor)
    assert second_batch.results[0].transcript_id == item.transcript_id
    assert len(list_transcripts_for_video(db, 'ok')) == 1

    terminal_source = 'https://www.youtube.com/@terminal/videos'
    discover(db, terminal_source, 'none', 'members', 'failed')
    def terminal_processor(url, **kwargs):
        video_id = url.split('=')[-1]
        if video_id == 'none': raise TubeScribeNoSubtitlesError('none')
        if video_id == 'members': raise TubeScribeMembersOnlyError('members')
        raise TubeScribeProcessingError('failed')
    terminal = process_pending_videos(db, raw, cleaned, limit=3, source_url=terminal_source, video_ids=['none','members','failed'], processor=terminal_processor)
    assert {result.status for result in terminal.results} == {'no_subtitles','members_only','failed'}
    assert all(get_transcript(db, video_id) is None for video_id in ('none','members','failed'))

    rollback_source = 'https://www.youtube.com/@rollback/videos'
    discover(db, rollback_source, 'rollback')
    connection = sqlite3.connect(db)
    connection.execute("CREATE TRIGGER reject_test_transcript BEFORE INSERT ON transcripts BEGIN SELECT RAISE(ABORT, 'test rejection'); END")
    connection.commit(); connection.close()
    try:
        process_pending_videos(db, raw, cleaned, source_url=rollback_source, video_ids=['rollback'], processor=success_processor)
    except StateStorageError:
        pass
    else:
        raise AssertionError('transcript write failure unexpectedly succeeded')
    connection = sqlite3.connect(db)
    try:
        assert connection.execute("SELECT status, transcript_id FROM processing_records WHERE video_id='rollback'").fetchone() == ('pending', None)
    finally:
        connection.close()
    assert get_transcript(db, 'rollback') is None
print('Processing integration checks passed.')


Processing integration checks passed.
